In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
import pandas as pd
df = pd.read_csv("/content/flight_delays[1].csv")
df

,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance
0,1,United,4558,ORD,MIA,2024-09-01 08:11,2024-09-01 08:30,2024-09-01 12:11,2024-09-01 12:19,8,Weather,True,False,Boeing 737,N71066,1031.0
1,2,Delta,8021,LAX,MIA,2024-09-01 10:25,2024-09-01 10:41,2024-09-01 13:25,2024-09-01 13:27,2,Air Traffic Control,True,True,Airbus A320,N22657,1006.0
2,3,Southwest,7520,DFW,SFO,2024-09-01 16:53,2024-09-01 17:05,2024-09-01 17:53,2024-09-01 18:07,14,Weather,True,True,Boeing 737,N95611,2980.0
3,4,Delta,2046,ORD,BOS,2024-09-01 14:44,2024-09-01 15:04,2024-09-01 18:44,2024-09-01 18:34,-10,NaN,False,False,Boeing 777,N90029,1408.0
4,5,Delta,6049,LAX,SEA,2024-09-01 01:51,2024-09-01 02:08,2024-09-01 05:51,2024-09-01 06:15,24,Air Traffic Control,False,True,Boeing 737,N27417,2298.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610076,610077,Delta,4649,DFW,BOS,2024-09-01 22:40,2024-09-01 23:00,2024-09-02 04:40,2024-09-02 05:09,29,Maintenance,False,False,Airbus A320,N44930,2321.0
610077,610078,United,6575,DFW,SEA,2024-09-01 03:04,2024-09-01 03:09,2024-09-01 04:04,2024-09-01 03:56,-8,NaN,False,True,Boeing 737,N83942,1752.0
610078,610079,Delta,5384,LAX,BOS,2024-09-01 16:41,2024-09-01 16:42,2024-09-01 17:41,2024-09-01 17:44,3,Maintenance,True,False,Airbus A320,N81416,2012.0
610079,610080,Southwest,8093,LAX,JFK,2024-09-01 15:14,2024-09-01 15:21,2024-09-01 17:14,2024-09-01 17:20,6,Maintenance,False,False,Airbus A320,N65669,789.0


In [5]:
print(df.head())
print(df.info)
print(df.columns)
print(df.isnull().sum())
print(df.describe)

   FlightID    Airline  FlightNumber Origin Destination ScheduledDeparture  \
0         1     United          4558    ORD         MIA   2024-09-01 08:11   
1         2      Delta          8021    LAX         MIA   2024-09-01 10:25   
2         3  Southwest          7520    DFW         SFO   2024-09-01 16:53   
3         4      Delta          2046    ORD         BOS   2024-09-01 14:44   
4         5      Delta          6049    LAX         SEA   2024-09-01 01:51   

    ActualDeparture  ScheduledArrival     ActualArrival  DelayMinutes  \
0  2024-09-01 08:30  2024-09-01 12:11  2024-09-01 12:19             8   
1  2024-09-01 10:41  2024-09-01 13:25  2024-09-01 13:27             2   
2  2024-09-01 17:05  2024-09-01 17:53  2024-09-01 18:07            14   
3  2024-09-01 15:04  2024-09-01 18:44  2024-09-01 18:34           -10   
4  2024-09-01 02:08  2024-09-01 05:51  2024-09-01 06:15            24   

           DelayReason  Cancelled  Diverted AircraftType TailNumber  Distance  
0           

In [6]:
df = df.drop(['ActualDeparture', 'ActualArrival'], axis=1)

In [7]:
df['DelayReason'] = df['DelayReason'].fillna("Unknown")
df = df.dropna()

In [14]:
# Process ScheduledDeparture if it exists
if 'ScheduledDeparture' in df.columns:
    df['ScheduledDeparture'] = pd.to_datetime(df['ScheduledDeparture'])
    df['hour'] = df['ScheduledDeparture'].dt.hour
    df['day'] = df['ScheduledDeparture'].dt.day
    df['month'] = df['ScheduledDeparture'].dt.month

# Process ScheduledArrival if it exists
if 'ScheduledArrival' in df.columns:
    df['ScheduledArrival'] = pd.to_datetime(df['ScheduledArrival'])
    df['arrival_hour'] = df['ScheduledArrival'].dt.hour
    df['arrival_day'] = df['ScheduledArrival'].dt.day
    df['arrival_month'] = df['ScheduledArrival'].dt.month

# Drop the original datetime columns if they were present
cols_to_drop_after_processing = []
if 'ScheduledDeparture' in df.columns:
    cols_to_drop_after_processing.append('ScheduledDeparture')
if 'ScheduledArrival' in df.columns:
    cols_to_drop_after_processing.append('ScheduledArrival')

if cols_to_drop_after_processing:
    df = df.drop(cols_to_drop_after_processing, axis=1)


In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Airline'] = le.fit_transform(df['Airline'])
df['Origin'] = le.fit_transform(df['Origin'])
df['Destination'] = le.fit_transform(df['Destination'])
df['DelayReason'] = le.fit_transform(df['DelayReason'])
df['AircraftType'] = le.fit_transform(df['AircraftType'])
df['TailNumber'] = le.fit_transform(df['TailNumber'])

In [19]:
X = df.drop(['DelayMinutes'], axis=1)
y = df['DelayMinutes']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
df = df.sample(n=100000, random_state=42)

In [27]:
model = RandomForestRegressor(n_estimators=20)

In [28]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(X_train, y_train)

DecisionTreeRegressor()

In [31]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)

[ 27.  14. -10. ...  26.   4.  22.]


In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

MAE: 8.366706005769736
MSE: 117.33685746131655
